# Inductive Conformal Prediction


Traditional machine learning models output a point value (regression) or class (classification) but do not provide rigorous guarantees about uncertainty.

Inductive conformal prediction (CP) addresses this limitation by transforming model outputs into prediction intervals (regression) or sets (classification) with statistical coverage guarantees. For a chosen confidence level (e.g. 95%), CP guarantees that the true label will be contained in the prediction set at least 95% of the time, assuming data sets are exchangeable (*marginal coverage guarantee*). Exchangability is a weaker assumption than independent and identically distributed (i.i.d.) data, which is typically assumed in machine learning.

The theoretical part of the notebook is based on [this publication](https://arxiv.org/abs/2107.07511). For more in-depth explanation, we refer the reder therefore to Angelopoulos and Bates.

In this notebook we:

1. Generate molecular descriptors from SMILES structures.
2. Train a Random Forest classifier.
3. Calibrate the model using a separate calibration dataset.
4. Compute non-conformity scores.
5. Construct conformal prediction sets.
6. Evaluate how uncertainty changes the interpretation of model predictions.

In [1]:
import pandas as pd
from morgoth.conformal_prediction import get_pred_score_true_class, eval_classification_true_class
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from rdkit.Chem import Descriptors, MolFromSmiles
import matplotlib.pyplot as plt
import math
import warnings
warnings.filterwarnings(action='ignore')
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

The dataset contains molecular structures represented as SMILES strings. These structures are converted into numerical molecular descriptors which serve as input features for machine learning.

In [3]:
smiles_df = pd.read_csv('data/compounds_with_smiles.csv', sep = '\t')
smiles_df.head()

,compound,smiles,inchi,inchikey
0,DTXSID9046499,CCOC(=O)COc1ccc2c(C)c(CCN(CC)CC)c(=O)oc2c1Cl.Cl,InChI=1S/C20H26ClNO5.ClH/c1-5-22(6-2)11-10-15-...,CCCZJRFQJNGCCU-UHFFFAOYSA-N
1,DTXSID3049386,CCC[N+]1(C)CCCCC1.F[P-](F)(F)(F)(F)F,InChI=1S/C9H20N.F6P/c1-3-7-10(2)8-5-4-6-9-10;1...,CDBFWKDOZHJELY-UHFFFAOYSA-N
2,DTXSID1048625,C[C@H](CSC(=O)c1ccccc1)C(=O)N1C[C@@H](Sc2ccccc...,InChI=1S/2C22H23NO4S2.Ca/c2*1-15(14-28-22(27)1...,NSYUKKYYVFVMST-LETVYOFWSA-L
3,DTXSID3045568,CCN(CC)Cc1cc(Nc2ccnc3cc(Cl)ccc23)ccc1O.Cl.Cl.O.O,InChI=1S/C20H22ClN3O.2ClH.2H2O/c1-3-24(4-2)13-...,YVNAYSHNIILOJS-UHFFFAOYSA-N
4,DTXSID6026298,Cc1cccc(C)c1,"InChI=1S/C8H10/c1-7-4-3-5-8(2)6-7/h3-6H,1-2H3",IVSZLXZYQVIEFR-UHFFFAOYSA-N


### Molecular Descriptor Generation

Most machine learning algorithms cannot directly process molecular structures.
We therefore convert each molecule into a vector of physicochemical properties
using RDKit.

Examples include:

- Molecular weight
- Topological indices
- Electrotopological descriptors
- Ring counts
- Functional group counts

The resulting feature matrix contains over 200 descriptors per compound.

In [ ]:
descriptors = {}
for compound, smiles in smiles_df.loc[:, ['compound', 'smiles']].values:
    try:
        mol = MolFromSmiles(smiles)
    except:
        continue
    descriptors[compound] = Descriptors.CalcMolDescriptors(mol)

In [ ]:
physchem_properties = pd.DataFrame(data = descriptors)
physchem_properties = physchem_properties.transpose()
physchem_properties = physchem_properties.astype('Float32')
physchem_properties.dropna(inplace=True, axis=1)
physchem_properties.head()

### Dataset Splitting

Inductive CP requires three distinct datasets:

- Training set:
  used to fit the machine learning model

- Calibration set:
  used to estimate uncertainty and compute thresholds

- Test set:
  used only for final evaluation

Separating calibration from training is the key distinction between
inductive conformal prediction and standard model evaluation.

In [ ]:
with open('data/train.txt', 'r') as train_compound_file:
    train_compounds = train_compound_file.read().splitlines()
with open('data/calibration.txt', 'r') as cal_compound_file:
    cal_compounds = cal_compound_file.read().splitlines()
with open('data/test.txt', 'r') as test_compound_file:
    test_compounds = test_compound_file.read().splitlines()

In [ ]:
splits = [
    len(train_compounds),
    len(cal_compounds),
    len(test_compounds)
]

plt.figure(figsize=(6,4))
plt.bar(
    ["Train", "Calibration", "Test"],
    splits
)

plt.ylabel("Number of Compounds")
plt.title("Dataset Partitioning")
plt.show()


In [ ]:
X_train = physchem_properties.loc[train_compounds, :]
X_cal = physchem_properties.loc[cal_compounds, :]
X_test = physchem_properties.loc[test_compounds, :]

In [ ]:
binary_response = pd.read_csv('data/binary_response.tsv', sep = '\t', index_col=0)
y_train = binary_response.loc[train_compounds, :]
y_test = binary_response.loc[test_compounds, :]
y_cal = binary_response.loc[cal_compounds, :]

### Random Forest Classifier

A Random Forest is an ensemble method that combines many decision trees.

Advantages:

- Handles high-dimensional descriptor spaces well.
- Captures nonlinear relationships.
- Robust to noisy descriptors.
- Provides class probabilities that can be used by conformal prediction.

The model is trained only on the training set.

In [ ]:
rfc = RandomForestClassifier(n_estimators=500, max_depth=20, min_samples_leaf=15, random_state=42, class_weight='balanced')

In [ ]:
rfc.fit(X_train, y_train)

### Calibration

After training, predictions are generated for the calibration dataset.

For every calibration sample we compute a (non-)conformity score
that measures how compatible the observed class is with the model prediction. In our case, we compute a non-conformity score, which is called true class ($TC$) score, and that measures the probability of being wrong for a particular sample.

These non-conformity scores form an empirical reference distribution.

The conformal threshold ($\hat{q}$) is derived from a quantile of this distribution
and determines how prediction sets are constructed for unseen compounds.

#### Intuition behind the CP Sets

The RF produces a probability for each class. Rather than directly using these probabilities as confidence estimates, we define a *non-conformity score* as the probability that a prediction is wrong.

For a given sample \(x_i\), the non-conformity score is based on the true class $c_i$ and its corresponding prediction probability \(P(c_i)\)

$TC({x_i}) = 1 - P(c_i)$


A low score corresponds to being correct with a high probabiliyt, while a high score means that the model was wrong with a high probability.

During calibration, we determine a threshold ($\hat{q}$) corresponding to the maximum probability of being wrong that can be tolerated, while still achieving the desired coverage guarantee. This threshold is estimated from the calibration set and represents the largest non-conformity score that is acceptable for a prediction to be considered reliable.

In [ ]:
minimal_certainty = 0.95
true_class_scores = get_pred_score_true_class(estimator=rfc, X=X_cal, y=y_cal.values, minimal_certainty=minimal_certainty, class_names=[0,1])

In [ ]:
n = len(cal_compounds)
q_class = np.quantile(true_class_scores, math.ceil((n+1)*minimal_certainty)/n)

In [ ]:
plt.figure(figsize=(7,4))

plt.hist(
    true_class_scores,
    bins=20,
    color="steelblue",
    edgecolor="black"
)

plt.axvline(
    q_class,
    color="red",
    linestyle="--",
    label=f"95% threshold = {q_class:.3f}"
)

plt.xlabel("Conformity Score")
plt.ylabel("Count")
plt.title("True Class Score Distribution")
plt.legend()
plt.show()

### Test predictions

For a new compound, we construct the conformal prediction set by including every class whose non-conformity score is below this threshold:

$\{ c : 1-P(c) \le \hat{q} \}$

Intuitively, this means:

- Classes with a sufficiently low probability of being wrong are retained.
- Classes with a probability of being wrong that exceeds the calibrated threshold are excluded.
- If only one class satisfies the criterion, a singleton prediction set is produced.
- If multiple classes satisfy the criterion, the model expresses uncertainty by returning all plausible classes.
- In rare cases no class may satisfy the criterion, resulting in an empty prediction set.

The threshold is chosen such that, on average, the true class will be retained in the prediction set with at least the specified coverage level (e.g. 95%). Therefore, conformal prediction does not aim to identify the single most likely class. Instead, it identifies all classes whose probability of being wrong is sufficiently small to maintain the desired statistical guarantee.


As we consider binary classification, prediction sets can have three outcomes:

- Size 0 (only possible if $\hat{q} < 0.5$):
  insufficient support

- Size 1:
  confident prediction

- Size 2 only possible if $\hat{q} >= 0.5$):
  ambiguous prediction; both classes remain plausible

The more single-class sets are output, i.e., prediction sets containing exactly one class, the more certain is the model and the more _efficient_ is the CP score.

#### Efficiency 
In classification, efficiency is commonly quantified as the fraction of single-class prediction sets over all predictions.

An efficient conformal predictor produces many single-class predictions, while still maintaining the desired ceratinty guarantee, i.e., _coverage_. Prediction sets containing multiple classes indicate uncertainty and are, therefore, less informative.

In [ ]:
test_predictions = rfc.predict_proba(X_test)

In [ ]:
results = eval_classification_true_class(y_pred_proba=test_predictions, y_test=y_test.values.flatten(), sample_names=y_test.index, minimal_certainty=minimal_certainty, q = q_class, num_classes=2, class_names=[0,1])

In [ ]:
prediction_set_sizes = (
    results["conformal_prediction_95_quantile"]
    .apply(len)
)

ax = prediction_set_sizes.value_counts().sort_index().plot(
    kind="bar",
    figsize=(6,4)
)

# efficiency = fraction of singleton prediction sets
singleton_ratio = len(prediction_set_sizes.loc[prediction_set_sizes == 1])/len(prediction_set_sizes)

ax.text(
    0.95, 0.95,
    f"Efficiency = {singleton_ratio:.1%}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        alpha=0.8
    )
)

plt.xlabel("Prediction Set Size")
plt.ylabel("Number of Compounds")
plt.title("Conformal Prediction Set Sizes")
plt.show()


#### Coverage

Coverage measures the validity of the conformal predictor. A prediction is considered covered if the true class is contained within the conformal prediction set. For example, if the prediction set is [0, 1], both classes are considered plausible and the prediction is counted as covered regardless of the true outcome.

The empirical coverage is calculated as the proportion of test compounds whose true label is included in the corresponding conformal prediction set. For a conformal predictor with a minimal certainty level of 95%, the empirical coverage should be close to or above 95%, demonstrating that the uncertainty estimates are well calibrated.

High coverage indicates that the conformal prediction sets reliably contain the true class, while lower-than-expected coverage may suggest violations of the assumptions underlying conformal prediction or insufficient calibration data.

In [ ]:
cp_sets = results["conformal_prediction_95_quantile"]

coverage = np.mean([
    y in pred_set
    for y, pred_set in zip(
        results["actual"],
        cp_sets
    )
])

not_covered = 1 - coverage

fig, ax = plt.subplots(figsize=(6,6))

ax.pie(
    [coverage, not_covered],
    labels=[
        f"Covered\n({coverage:.1%})",
        f"Not covered\n({not_covered:.1%})"
    ],
    colors=["steelblue", "lightcoral"],
    autopct="%1.1f%%",
    startangle=90
)

ax.set_title(
    f"Empirical Coverage\nCertainty level = {minimal_certainty:.0%}"
)

plt.show()

### Conclusions

The RF classifier provides probability estimates for binary classification.

Inductive CP converts these probabilities into statistically
valid prediction sets by calibrating uncertainty on an independent calibration set.

Benefits:

- Rigorous coverage guarantees.
- Explicit uncertainty quantification.
- Improved reliability compared with raw probabilities.

Limitations:

- Requires a dedicated calibration set.
- Larger uncertainty leads to less specific predictions.
- Not really suited for small datasets $\rightarrow$ transductive CP is needed

Overall, CP provides a practical framework for deploying trustworthy
machine learning models in cheminformatics applications.